In [1]:
# Install required packages
!pip install -q transformers datasets torch accelerate sentencepiece

In [2]:
import json
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch import nn
from tqdm.auto import tqdm
import os
import re
from typing import List, Dict, Tuple
from collections import Counter
import warnings
import shutil
warnings.filterwarnings('ignore')

2026-02-09 06:30:19.461803: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770618619.803259      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770618619.914316      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770618620.755791      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770618620.755834      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770618620.755843      55 computation_placer.cc:177] computation placer alr

In [3]:
# Configuration with TIER 1 improvements
class Config:
    # Paths
    TRAIN_ZIP = '/kaggle/input/dimasqp/subtask3'
    DATA_DIR = './subtask3_data'
    OUTPUT_DIR = './subtask3_outputs'
    CHECKPOINT_DIR = './subtask3_checkpoints'
    
    # Model
    MODEL_NAME = 't5-base'
    MAX_INPUT_LENGTH = 128
    MAX_OUTPUT_LENGTH = 320
    
    # Training
    BATCH_SIZE = 12
    EPOCHS = 20
    LEARNING_RATE = 3e-4
    LABEL_SMOOTHING = 0.1
    WARMUP_RATIO = 0.1
    WEIGHT_DECAY = 0.01
    GRADIENT_CLIP = 1.0
    GRADIENT_ACCUMULATION_STEPS = 2
    
    # VA normalization
    VA_MIN = 1.0
    VA_MAX = 9.0
    VA_THRESHOLD = 2.0
    
    # IMPROVEMENT A4: Increased beam search
    NUM_BEAMS = 6
    
    # Device
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Random seed
    SEED = 42
    
    # Early stopping
    PATIENCE = 3
    
    # IMPROVEMENT A2: Progressive VA weighting schedule
    VA_WEIGHT_SCHEDULE = {
        'start': 0.3,
        'middle': 0.5,
        'end': 0.7
    }
    
    # IMPROVEMENT A3: VA diversity loss weight
    VA_DIVERSITY_WEIGHT = 0.1
    
    # IMPROVEMENT A1: Span matching threshold
    SPAN_MATCH_THRESHOLD = 0.75

config = Config()
os.makedirs(config.DATA_DIR, exist_ok=True)
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)

files_to_copy = [
    "eng_laptop_dev_task3.jsonl",
    "eng_laptop_test_task3.jsonl",
    "eng_laptop_train_alltasks.jsonl",
    "eng_restaurant_dev_task3.jsonl",
    "eng_restaurant_test_task3.jsonl",
    "eng_restaurant_train_alltasks.jsonl"
]

for fname in files_to_copy:
    src = os.path.join(config.TRAIN_ZIP, fname)
    dst = os.path.join(config.DATA_DIR, fname)
    shutil.copy(src, dst)
    print(f"Copied {fname}")

torch.manual_seed(config.SEED)
np.random.seed(config.SEED)

print(f"\nDevice: {config.DEVICE}")
print(f"Learning Rate: {config.LEARNING_RATE}")
print(f"Label Smoothing: {config.LABEL_SMOOTHING}")
print(f"Beam Search: {config.NUM_BEAMS}")
print(f"VA Weight Schedule: {config.VA_WEIGHT_SCHEDULE}")
print(f"Span Match Threshold: {config.SPAN_MATCH_THRESHOLD}")

Copied eng_laptop_dev_task3.jsonl
Copied eng_laptop_test_task3.jsonl
Copied eng_laptop_train_alltasks.jsonl
Copied eng_restaurant_dev_task3.jsonl
Copied eng_restaurant_test_task3.jsonl
Copied eng_restaurant_train_alltasks.jsonl

Device: cuda
Learning Rate: 0.0003
Label Smoothing: 0.1
Beam Search: 6
VA Weight Schedule: {'start': 0.3, 'middle': 0.5, 'end': 0.7}
Span Match Threshold: 0.75


In [4]:
# Data loading and category extraction
def load_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

def extract_all_categories(train_files):
    categories = set()
    category_counts = Counter()
    
    for file_path in train_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                data = json.loads(line)
                for quad in data.get('Quadruplet', []):
                    if quad['Aspect'] != 'NULL' and quad['Opinion'] != 'NULL':
                        cat = quad['Category']
                        categories.add(cat)
                        category_counts[cat] += 1
    
    sorted_cats = [cat for cat, _ in category_counts.most_common()]
    return sorted_cats, category_counts

def parse_va(va_string):
    v, a = va_string.split('#')
    return float(v), float(a)

def format_va(valence, arousal):
    v = np.clip(valence, config.VA_MIN, config.VA_MAX)
    a = np.clip(arousal, config.VA_MIN, config.VA_MAX)
    return f"{v:.2f}#{a:.2f}"

print("\nExtracting categories from training data...")
train_files = [
    f'{config.DATA_DIR}/eng_restaurant_train_alltasks.jsonl',
    f'{config.DATA_DIR}/eng_laptop_train_alltasks.jsonl'
]

VALID_CATEGORIES, category_freq = extract_all_categories(train_files)
MOST_FREQUENT_CATEGORY = VALID_CATEGORIES[0]

print(f"Total categories found: {len(VALID_CATEGORIES)}")
print(f"Most frequent category: {MOST_FREQUENT_CATEGORY}")
print(f"\nTop 20 categories:")
for i, cat in enumerate(VALID_CATEGORIES[:20], 1):
    print(f"  {i:2d}. {cat:40s} ({category_freq[cat]} occurrences)")


Extracting categories from training data...
Total categories found: 122
Most frequent category: FOOD#QUALITY

Top 20 categories:
   1. FOOD#QUALITY                             (1034 occurrences)
   2. LAPTOP#GENERAL                           (639 occurrences)
   3. SERVICE#GENERAL                          (431 occurrences)
   4. AMBIENCE#GENERAL                         (327 occurrences)
   5. LAPTOP#DESIGN_FEATURES                   (288 occurrences)
   6. LAPTOP#OPERATION_PERFORMANCE             (226 occurrences)
   7. RESTAURANT#GENERAL                       (202 occurrences)
   8. FOOD#STYLE_OPTIONS                       (141 occurrences)
   9. KEYBOARD#GENERAL                         (133 occurrences)
  10. LAPTOP#QUALITY                           (126 occurrences)
  11. DISPLAY#GENERAL                          (123 occurrences)
  12. LAPTOP#PRICE                             (104 occurrences)
  13. DISPLAY#DESIGN_FEATURES                  (104 occurrences)
  14. KEYBOARD#OPERATION

In [5]:
# IMPROVEMENT A1: Span Validation Functions
def span_exists_in_text(span, text, threshold=0.75):
    """
    Check if a span exists in the text using fuzzy matching.
    """
    if not span or not text:
        return False
    
    span_lower = span.lower().strip()
    text_lower = text.lower()
    
    # Exact substring match
    if span_lower in text_lower:
        return True
    
    # Multi-word fuzzy match
    span_words = span_lower.split()
    if len(span_words) > 1:
        matches = sum(1 for word in span_words if word in text_lower)
        match_ratio = matches / len(span_words)
        return match_ratio >= threshold
    
    # Single word partial match
    return any(span_lower in word for word in text_lower.split())

def validate_spans(aspect, opinion, text, threshold=0.75):
    """
    Validate that both aspect and opinion exist in the text.
    """
    aspect_valid = span_exists_in_text(aspect, text, threshold)
    opinion_valid = span_exists_in_text(opinion, text, threshold)
    
    if not aspect_valid and not opinion_valid:
        return False, "both_missing"
    elif not aspect_valid:
        return False, "aspect_missing"
    elif not opinion_valid:
        return False, "opinion_missing"
    else:
        return True, "valid"

print("Span validation functions loaded")
print(f"Span match threshold: {config.SPAN_MATCH_THRESHOLD}")

Span validation functions loaded
Span match threshold: 0.75


In [6]:
# IMPROVEMENT A5: Enhanced Category Validation
def validate_category(category, valid_categories, most_frequent_category):
    """
    Enhanced category validation with better fuzzy matching.
    """
    if not category:
        return most_frequent_category
    
    # 1. Exact match
    if category in valid_categories:
        return category
    
    # 2. Case-insensitive exact match
    category_upper = category.upper()
    for valid_cat in valid_categories:
        if category_upper == valid_cat.upper():
            return valid_cat
    
    # 3. Substring match
    for valid_cat in valid_categories:
        if category_upper in valid_cat.upper() or valid_cat.upper() in category_upper:
            return valid_cat
    
    # 4. Partial match by splitting on '#'
    if '#' in category:
        cat_parts = category.upper().split('#')
        main_category = cat_parts[0]
        
        candidates = [c for c in valid_categories if c.upper().startswith(main_category)]
        if candidates:
            for c in candidates:
                if 'GENERAL' in c.upper():
                    return c
            return candidates[0]
    
    # 5. Keyword-based matching with domain awareness
    category_upper = category.upper()
    
    laptop_keywords = {
        'LAPTOP': 'LAPTOP#GENERAL',
        'COMPUTER': 'LAPTOP#GENERAL',
        'BATTERY': 'BATTERY#OPERATION_PERFORMANCE',
        'SCREEN': 'DISPLAY#QUALITY',
        'DISPLAY': 'DISPLAY#QUALITY',
        'KEYBOARD': 'KEYBOARD#QUALITY',
        'TOUCHPAD': 'MOUSE#USABILITY',
        'TRACKPAD': 'MOUSE#USABILITY',
        'CPU': 'CPU#OPERATION_PERFORMANCE',
        'PROCESSOR': 'CPU#OPERATION_PERFORMANCE',
        'MEMORY': 'MEMORY#OPERATION_PERFORMANCE',
        'RAM': 'MEMORY#OPERATION_PERFORMANCE',
        'HARD_DRIVE': 'HARD_DISC#OPERATION_PERFORMANCE',
        'STORAGE': 'HARD_DISC#OPERATION_PERFORMANCE',
        'FAN': 'FANS_COOLING#OPERATION_PERFORMANCE',
        'COOLING': 'FANS_COOLING#OPERATION_PERFORMANCE',
        'PORT': 'PORTS#CONNECTIVITY',
        'USB': 'PORTS#CONNECTIVITY'
    }
    
    restaurant_keywords = {
        'FOOD': 'FOOD#QUALITY',
        'DRINK': 'DRINKS#QUALITY',
        'BEVERAGE': 'DRINKS#QUALITY',
        'SERVICE': 'SERVICE#GENERAL',
        'WAITER': 'SERVICE#GENERAL',
        'STAFF': 'SERVICE#GENERAL',
        'AMBIENCE': 'AMBIENCE#GENERAL',
        'ATMOSPHERE': 'AMBIENCE#GENERAL',
        'DECOR': 'AMBIENCE#GENERAL',
        'RESTAURANT': 'RESTAURANT#GENERAL',
        'PLACE': 'RESTAURANT#GENERAL',
        'LOCATION': 'LOCATION#GENERAL',
        'PRICE': 'RESTAURANT#PRICES',
        'COST': 'RESTAURANT#PRICES'
    }
    
    for keyword, target_cat in laptop_keywords.items():
        if keyword in category_upper:
            for valid_cat in valid_categories:
                if target_cat in valid_cat or valid_cat.startswith(target_cat.split('#')[0]):
                    return valid_cat
    
    for keyword, target_cat in restaurant_keywords.items():
        if keyword in category_upper:
            for valid_cat in valid_categories:
                if target_cat in valid_cat or valid_cat.startswith(target_cat.split('#')[0]):
                    return valid_cat
    
    return most_frequent_category

print("Enhanced category validation loaded")

Enhanced category validation loaded


In [7]:
# Format conversion with span validation
def quadruples_to_sequence(quadruples):
    if not quadruples:
        return "none"
    
    parts = []
    for quad in quadruples:
        aspect = quad['Aspect']
        category = quad['Category']
        opinion = quad['Opinion']
        va = quad['VA']
        parts.append(f"[A]{aspect}[C]{category}[O]{opinion}[V]{va}")
    
    return "[SEP]".join(parts)

def sequence_to_quadruples(sequence, original_text=None):
    """
    Parse with IMPROVEMENT A1: Span Validation
    """
    if sequence.strip() == "none":
        return []
    
    quadruples = []
    pattern = r'\[A\](.*?)\[C\](.*?)\[O\](.*?)\[V\](.*?)(?:\[SEP\]|$)'
    matches = re.finditer(pattern, sequence)
    
    for match in matches:
        aspect = match.group(1).strip()
        category = match.group(2).strip()
        opinion = match.group(3).strip()
        va = match.group(4).strip()
        
        # IMPROVEMENT A1: Validate spans
        if original_text:
            is_valid, reason = validate_spans(
                aspect, 
                opinion, 
                original_text, 
                config.SPAN_MATCH_THRESHOLD
            )
            if not is_valid:
                continue
        
        # IMPROVEMENT A5: Enhanced category validation
        category = validate_category(category, VALID_CATEGORIES, MOST_FREQUENT_CATEGORY)
        
        if '#' in va:
            try:
                v, a = parse_va(va)
                va = format_va(v, a)
                quadruples.append({
                    'Aspect': aspect,
                    'Category': category,
                    'Opinion': opinion,
                    'VA': va
                })
            except:
                continue
    
    return quadruples

print("Sequence parsing with span validation loaded")

Sequence parsing with span validation loaded


In [8]:
# Dataset with NULL filtering
class DimASQPDataset(Dataset):
    def __init__(self, data, tokenizer, max_input_length, max_output_length):
        self.samples = []
        self.tokenizer = tokenizer
        self.max_input_length = max_input_length
        self.max_output_length = max_output_length
        
        total_quads = 0
        null_quads = 0
        valid_quads = 0
        
        for item in data:
            text = item['Text']
            quads = item.get('Quadruplet', [])
            
            valid_quad_list = []
            for quad in quads:
                total_quads += 1
                if quad['Aspect'] != 'NULL' and quad['Opinion'] != 'NULL':
                    valid_quad_list.append(quad)
                    valid_quads += 1
                else:
                    null_quads += 1
            
            if valid_quad_list:
                self.samples.append({
                    'text': text,
                    'quadruples': valid_quad_list,
                    'id': item.get('ID', '')
                })
        
        print(f"  Total quadruplets: {total_quads}")
        print(f"  NULL quadruplets: {null_quads} ({100*null_quads/total_quads:.1f}%)")
        print(f"  Valid quadruplets: {valid_quads} ({100*valid_quads/total_quads:.1f}%)")
        print(f"  Valid samples: {len(self.samples)} (from {len(data)} total)")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        text = sample['text']
        quadruples = sample['quadruples']
        
        input_text = f"extract quadruples: {text}"
        target_text = quadruples_to_sequence(quadruples)
        
        input_encoding = self.tokenizer(
            input_text,
            max_length=self.max_input_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        target_encoding = self.tokenizer(
            target_text,
            max_length=self.max_output_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        labels = target_encoding['input_ids'].squeeze(0)
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            'input_ids': input_encoding['input_ids'].squeeze(0),
            'attention_mask': input_encoding['attention_mask'].squeeze(0),
            'labels': labels,
            'id': sample['id'],
            'text': text
        }

class DimASQPTestDataset(Dataset):
    def __init__(self, data, tokenizer, max_input_length):
        self.data = data
        self.tokenizer = tokenizer
        self.max_input_length = max_input_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        text = item['Text']
        
        input_text = f"extract quadruples: {text}"
        
        input_encoding = self.tokenizer(
            input_text,
            max_length=self.max_input_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': input_encoding['input_ids'].squeeze(0),
            'attention_mask': input_encoding['attention_mask'].squeeze(0),
            'id': item['ID'],
            'text': text
        }

In [9]:
# Continuous F1 metric
def continuous_f1_score(predictions, references, va_threshold=2.0):
    """
    Calculate Continuous F1 for DimASQP.
    """
    
    def calculate_va_penalty(pred_va, gold_va, threshold):
        pred_v, pred_a = parse_va(pred_va)
        gold_v, gold_a = parse_va(gold_va)
        
        va_error = np.sqrt((pred_v - gold_v)**2 + (pred_a - gold_a)**2)
        penalty = min(1.0, va_error / threshold)
        
        return 1.0 - penalty
    
    def quad_matches(pred, gold):
        if pred['Category'] != gold['Category']:
            return False
        
        aspect_match = (
            pred['Aspect'] == gold['Aspect'] or
            pred['Aspect'] in gold['Aspect'] or
            gold['Aspect'] in pred['Aspect']
        )
        if not aspect_match:
            return False
        
        opinion_match = (
            pred['Opinion'] == gold['Opinion'] or
            pred['Opinion'] in gold['Opinion'] or
            gold['Opinion'] in pred['Opinion']
        )
        if not opinion_match:
            return False
        
        return True
    
    pred_scores = []
    for pred in predictions:
        best_score = 0.0
        for gold in references:
            if quad_matches(pred, gold):
                va_score = calculate_va_penalty(pred['VA'], gold['VA'], va_threshold)
                best_score = max(best_score, va_score)
        pred_scores.append(best_score)
    
    ref_scores = []
    for gold in references:
        best_score = 0.0
        for pred in predictions:
            if quad_matches(pred, gold):
                va_score = calculate_va_penalty(pred['VA'], gold['VA'], va_threshold)
                best_score = max(best_score, va_score)
        ref_scores.append(best_score)
    
    if len(predictions) > 0:
        cPrecision = sum(pred_scores) / len(predictions)
    else:
        cPrecision = 0.0
    
    if len(references) > 0:
        cRecall = sum(ref_scores) / len(references)
    else:
        cRecall = 0.0
    
    if cPrecision + cRecall > 0:
        cF1 = 2 * (cPrecision * cRecall) / (cPrecision + cRecall)
    else:
        cF1 = 0.0
    
    v_errors = []
    a_errors = []
    va_values = []
    
    for pred in predictions:
        for gold in references:
            if quad_matches(pred, gold):
                pred_v, pred_a = parse_va(pred['VA'])
                gold_v, gold_a = parse_va(gold['VA'])
                
                v_errors.append((pred_v - gold_v) ** 2)
                a_errors.append((pred_a - gold_a) ** 2)
                va_values.append([pred_v, pred_a])
    
    va_rmse_v = np.sqrt(np.mean(v_errors)) if v_errors else float('inf')
    va_rmse_a = np.sqrt(np.mean(a_errors)) if a_errors else float('inf')
    va_variance = np.var(va_values) if va_values else 0.0
    
    return {
        'cF1': cF1,
        'cPrecision': cPrecision,
        'cRecall': cRecall,
        'VA_RMSE_V': va_rmse_v,
        'VA_RMSE_A': va_rmse_a,
        'VA_Variance': va_variance,
        'num_predictions': len(predictions),
        'num_references': len(references)
    }

In [10]:
# Load data
print("\nLoading data...")
train_rest = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_train_alltasks.jsonl')
train_laptop = load_jsonl(f'{config.DATA_DIR}/eng_laptop_train_alltasks.jsonl')
dev_rest = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_dev_task3.jsonl')
dev_laptop = load_jsonl(f'{config.DATA_DIR}/eng_laptop_dev_task3.jsonl')

train_data = train_rest + train_laptop
dev_data = dev_rest + dev_laptop

print(f"Raw train samples: {len(train_data)}")
print(f"Raw dev samples: {len(dev_data)}")

print("\nLoading tokenizer and model...")
tokenizer = T5Tokenizer.from_pretrained(config.MODEL_NAME)

print("\nCreating training dataset (with NULL filtering):")
train_dataset = DimASQPDataset(
    train_data,
    tokenizer,
    config.MAX_INPUT_LENGTH,
    config.MAX_OUTPUT_LENGTH
)

print("\nCreating dev dataset (with NULL filtering):")
dev_dataset = DimASQPDataset(
    dev_data,
    tokenizer,
    config.MAX_INPUT_LENGTH,
    config.MAX_OUTPUT_LENGTH
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True
)
dev_loader = DataLoader(
    dev_dataset,
    batch_size=config.BATCH_SIZE
)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Dev batches: {len(dev_loader)}")


Loading data...
Raw train samples: 6360
Raw dev samples: 400

Loading tokenizer and model...


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565



Creating training dataset (with NULL filtering):
  Total quadruplets: 9432
  NULL quadruplets: 3725 (39.5%)
  Valid quadruplets: 5707 (60.5%)
  Valid samples: 3732 (from 6360 total)

Creating dev dataset (with NULL filtering):
  Total quadruplets: 725
  NULL quadruplets: 0 (0.0%)
  Valid quadruplets: 725 (100.0%)
  Valid samples: 400 (from 400 total)

Train batches: 311
Dev batches: 34


In [11]:
# Training functions with IMPROVEMENTS A2 & A3
def get_va_weight(epoch, total_epochs):
    """
    IMPROVEMENT A2: Progressive VA weighting.
    """
    if epoch < total_epochs * 0.3:
        return config.VA_WEIGHT_SCHEDULE['start']
    elif epoch < total_epochs * 0.65:
        return config.VA_WEIGHT_SCHEDULE['middle']
    else:
        return config.VA_WEIGHT_SCHEDULE['end']

def train_epoch(model, dataloader, optimizer, scheduler, device, gradient_accumulation_steps, epoch, total_epochs):
    model.train()
    total_loss = 0
    
    optimizer.zero_grad()
    
    va_weight = get_va_weight(epoch, total_epochs)
    loss_fct = nn.CrossEntropyLoss(label_smoothing=config.LABEL_SMOOTHING, ignore_index=-100)
    
    for step, batch in enumerate(tqdm(dataloader, desc='Training')):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        logits = outputs.logits
        loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
        
        # IMPROVEMENT A3: VA diversity loss
        diversity_loss = torch.tensor(0.0, device=device)
        if step % 10 == 0:
            with torch.no_grad():
                pred_ids = torch.argmax(logits, dim=-1)
                unique_ratio = len(torch.unique(pred_ids)) / pred_ids.numel()
                if unique_ratio < 0.3:
                    diversity_loss = config.VA_DIVERSITY_WEIGHT * (0.3 - unique_ratio)
        
        total_batch_loss = loss + diversity_loss
        total_batch_loss = total_batch_loss / gradient_accumulation_steps
        total_batch_loss.backward()
        
        if (step + 1) % gradient_accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader), va_weight

def evaluate_with_metrics(model, dataloader, tokenizer, device):
    model.eval()
    total_loss = 0
    
    all_predictions = []
    all_references = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            texts = batch['text']
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            total_loss += outputs.loss.item()
            
            # IMPROVEMENT A4: Beam search 6
            generated = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=config.MAX_OUTPUT_LENGTH,
                num_beams=config.NUM_BEAMS,
                early_stopping=True
            )
            
            for i, gen_ids in enumerate(generated):
                pred_seq = tokenizer.decode(gen_ids, skip_special_tokens=True)
                # IMPROVEMENT A1: Span validation
                pred_quads = sequence_to_quadruples(pred_seq, texts[i])
                
                label_ids = labels[i].cpu().numpy()
                label_ids = label_ids[label_ids != -100]
                ref_seq = tokenizer.decode(label_ids, skip_special_tokens=True)
                ref_quads = sequence_to_quadruples(ref_seq)
                
                all_predictions.append(pred_quads)
                all_references.append(ref_quads)
    
    avg_loss = total_loss / len(dataloader)
    
    flat_preds = [q for quads in all_predictions for q in quads]
    flat_refs = [q for quads in all_references for q in quads]
    
    metrics = continuous_f1_score(flat_preds, flat_refs, config.VA_THRESHOLD)
    metrics['loss'] = avg_loss
    
    return metrics

print("Training functions loaded with TIER 1 improvements")

Training functions loaded with TIER 1 improvements


In [12]:
# Initialize model
model = T5ForConditionalGeneration.from_pretrained(config.MODEL_NAME).to(config.DEVICE)

optimizer = AdamW(
    model.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

total_steps = len(train_loader) * config.EPOCHS // config.GRADIENT_ACCUMULATION_STEPS
warmup_steps = int(total_steps * config.WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f"\nTotal steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


Total steps: 3110
Warmup steps: 311
Model parameters: 222.9M


In [13]:
# Training loop
print("\n" + "="*80)
print("Starting training with TIER 1 improvements...")
print("="*80)
print("\nImprovements active:")
print(f"  A1: Span Validation (threshold={config.SPAN_MATCH_THRESHOLD})")
print(f"  A2: Progressive VA Weight (start={config.VA_WEIGHT_SCHEDULE['start']}, middle={config.VA_WEIGHT_SCHEDULE['middle']}, end={config.VA_WEIGHT_SCHEDULE['end']})")
print(f"  A3: VA Diversity Loss (weight={config.VA_DIVERSITY_WEIGHT})")
print(f"  A4: Beam Search = {config.NUM_BEAMS}")
print("  A5: Enhanced Category Validation")
print("="*80)

best_cf1 = 0.0
best_loss = float('inf')
patience_counter = 0

for epoch in range(config.EPOCHS):
    print(f"\nEpoch {epoch+1}/{config.EPOCHS}")
    print("-" * 80)
    
    train_loss, va_weight = train_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        config.DEVICE,
        config.GRADIENT_ACCUMULATION_STEPS,
        epoch,
        config.EPOCHS
    )
    
    metrics = evaluate_with_metrics(model, dev_loader, tokenizer, config.DEVICE)
    
    print("\nResults:")
    print(f"  Train Loss:  {train_loss:.4f}")
    print(f"  Dev Loss:    {metrics['loss']:.4f}")
    print(f"  Precision:   {metrics['cPrecision']:.4f}")
    print(f"  Recall:      {metrics['cRecall']:.4f}")
    print(f"  F1 Score:    {metrics['cF1']:.4f}")
    print(f"  VA RMSE (V): {metrics['VA_RMSE_V']:.4f}")
    print(f"  VA RMSE (A): {metrics['VA_RMSE_A']:.4f}")
    print(f"  VA Variance: {metrics['VA_Variance']:.4f}")
    print(f"  VA Weight:   {va_weight:.2f} (epoch {epoch+1})")
    
    if metrics['cF1'] > best_cf1:
        best_cf1 = metrics['cF1']
        patience_counter = 0
        
        model.save_pretrained(f'{config.CHECKPOINT_DIR}/best_model')
        tokenizer.save_pretrained(f'{config.CHECKPOINT_DIR}/best_model')
        
        print(f"\n  New best cF1: {best_cf1:.4f}")
    else:
        patience_counter += 1
        print(f"\n  No improvement for {patience_counter} epoch(s)")
    
    if metrics['loss'] < best_loss:
        best_loss = metrics['loss']
    
    if patience_counter >= config.PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break
    
    print("=" * 80)

print(f"\nTraining completed!")
print(f"Best cF1: {best_cf1:.4f}")
print(f"Best Loss: {best_loss:.4f}")


Starting training with TIER 1 improvements...

Improvements active:
  A1: Span Validation (threshold=0.75)
  A2: Progressive VA Weight (start=0.3, middle=0.5, end=0.7)
  A3: VA Diversity Loss (weight=0.1)
  A4: Beam Search = 6
  A5: Enhanced Category Validation

Epoch 1/20
--------------------------------------------------------------------------------


Training:   0%|          | 0/311 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/34 [00:00<?, ?it/s]


Results:
  Train Loss:  3.3332
  Dev Loss:    0.5251
  Precision:   0.3076
  Recall:      0.1561
  F1 Score:    0.2071
  VA RMSE (V): 1.3131
  VA RMSE (A): 1.0358
  VA Variance: 0.0621
  VA Weight:   0.30 (epoch 1)

  New best cF1: 0.2071

Epoch 2/20
--------------------------------------------------------------------------------


Training:   0%|          | 0/311 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/34 [00:00<?, ?it/s]


Results:
  Train Loss:  1.6142
  Dev Loss:    0.4463
  Precision:   0.3736
  Recall:      0.2910
  F1 Score:    0.3272
  VA RMSE (V): 0.8278
  VA RMSE (A): 0.8695
  VA Variance: 0.8928
  VA Weight:   0.50 (epoch 9)

  No improvement for 2 epoch(s)

Epoch 10/20
--------------------------------------------------------------------------------


Training:   0%|          | 0/311 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/34 [00:00<?, ?it/s]


Results:
  Train Loss:  1.6057
  Dev Loss:    0.4541
  Precision:   0.3624
  Recall:      0.2766
  F1 Score:    0.3137
  VA RMSE (V): 0.8878
  VA RMSE (A): 0.9412
  VA Variance: 0.9566
  VA Weight:   0.50 (epoch 10)

  No improvement for 3 epoch(s)

Early stopping at epoch 10

Training completed!
Best cF1: 0.3275
Best Loss: 0.4164


In [14]:
# Load best model
print("\nLoading best model...")
model = T5ForConditionalGeneration.from_pretrained(
    f'{config.CHECKPOINT_DIR}/best_model'
).to(config.DEVICE)
tokenizer = T5Tokenizer.from_pretrained(f'{config.CHECKPOINT_DIR}/best_model')
print("Model loaded successfully")


Loading best model...
Model loaded successfully


In [15]:
# Prediction functions
def predict(model, dataloader, tokenizer, device):
    model.eval()
    all_predictions = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Predicting'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            texts = batch['text']
            
            generated = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=config.MAX_OUTPUT_LENGTH,
                num_beams=config.NUM_BEAMS,
                early_stopping=True
            )
            
            for i, gen_ids in enumerate(generated):
                sequence = tokenizer.decode(gen_ids, skip_special_tokens=True)
                quadruples = sequence_to_quadruples(sequence, texts[i])
                all_predictions.append(quadruples)
    
    return all_predictions

def save_predictions(predictions, original_data, output_file):
    with open(output_file, 'w', encoding='utf-8') as f:
        for pred, orig in zip(predictions, original_data):
            result = {
                'ID': orig['ID'],
                'Quadruplet': pred
            }
            f.write(json.dumps(result, ensure_ascii=False) + '\n')
    print(f"Predictions saved to {output_file}")

In [16]:
# Generate predictions
print("\nGenerating predictions for test sets...")

test_rest_data = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_test_task3.jsonl')
test_rest_dataset = DimASQPTestDataset(test_rest_data, tokenizer, config.MAX_INPUT_LENGTH)
test_rest_loader = DataLoader(test_rest_dataset, batch_size=config.BATCH_SIZE)
predictions_rest = predict(model, test_rest_loader, tokenizer, config.DEVICE)
save_predictions(predictions_rest, test_rest_data, f'{config.OUTPUT_DIR}/restaurant_test_predictions.jsonl')

test_laptop_data = load_jsonl(f'{config.DATA_DIR}/eng_laptop_test_task3.jsonl')
test_laptop_dataset = DimASQPTestDataset(test_laptop_data, tokenizer, config.MAX_INPUT_LENGTH)
test_laptop_loader = DataLoader(test_laptop_dataset, batch_size=config.BATCH_SIZE)
predictions_laptop = predict(model, test_laptop_loader, tokenizer, config.DEVICE)
save_predictions(predictions_laptop, test_laptop_data, f'{config.OUTPUT_DIR}/laptop_test_predictions.jsonl')

print("\nAll predictions completed!")


Generating predictions for test sets...


Predicting:   0%|          | 0/84 [00:00<?, ?it/s]

Predictions saved to ./subtask3_outputs/restaurant_test_predictions.jsonl


Predicting:   0%|          | 0/84 [00:00<?, ?it/s]

Predictions saved to ./subtask3_outputs/laptop_test_predictions.jsonl

All predictions completed!


In [18]:
# Verification with original text
print("\n" + "="*80)
print("Prediction Verification")
print("="*80)

# Load test data to get original texts
test_rest_data = load_jsonl(f'{config.DATA_DIR}/eng_restaurant_test_task3.jsonl')
test_laptop_data = load_jsonl(f'{config.DATA_DIR}/eng_laptop_test_task3.jsonl')

print("\nRestaurant test samples:")
with open(f'{config.OUTPUT_DIR}/restaurant_test_predictions.jsonl', 'r') as f:
    for i, line in enumerate(f):
        if i < 3:
            data = json.loads(line)
            # Get original text
            original_text = test_rest_data[i]['Text']
            
            print(f"\nSample {i+1}:")
            print(f"  ID: {data['ID']}")
            print(f"  Text: {original_text}")
            print(f"  Quadruplets: {len(data['Quadruplet'])}")
            if data['Quadruplet']:
                for j, q in enumerate(data['Quadruplet'][:3]):  # Show up to 3 quadruplets
                    print(f"  [{j+1}] A={q['Aspect']}, C={q['Category']}, O={q['Opinion']}, VA={q['VA']}")

print("\n" + "-"*80)
print("Laptop test samples:")
with open(f'{config.OUTPUT_DIR}/laptop_test_predictions.jsonl', 'r') as f:
    for i, line in enumerate(f):
        if i < 3:
            data = json.loads(line)
            # Get original text
            original_text = test_laptop_data[i]['Text']
            
            print(f"\nSample {i+1}:")
            print(f"  ID: {data['ID']}")
            print(f"  Text: {original_text}")
            print(f"  Quadruplets: {len(data['Quadruplet'])}")
            if data['Quadruplet']:
                for j, q in enumerate(data['Quadruplet'][:3]):  # Show up to 3 quadruplets
                    print(f"  [{j+1}] A={q['Aspect']}, C={q['Category']}, O={q['Opinion']}, VA={q['VA']}")

print("\n" + "-"*80)
print("Category distribution in LAPTOP predictions:")
laptop_categories = []
with open(f'{config.OUTPUT_DIR}/laptop_test_predictions.jsonl', 'r') as f:
    for line in f:
        data = json.loads(line)
        for quad in data.get('Quadruplet', []):
            laptop_categories.append(quad['Category'])

laptop_cat_counts = Counter(laptop_categories)
print(f"Total quadruples: {len(laptop_categories)}")
print(f"Unique categories: {len(laptop_cat_counts)}")
print(f"\nTop 10 categories:")
for cat, count in laptop_cat_counts.most_common(10):
    print(f"  {cat}: {count}")

restaurant_in_laptop = sum(1 for cat in laptop_categories if 'RESTAURANT' in cat)
print(f"\nRESTAURANT categories in laptop: {restaurant_in_laptop} ({100*restaurant_in_laptop/len(laptop_categories) if laptop_categories else 0:.1f}%)")
if laptop_categories and restaurant_in_laptop > len(laptop_categories) * 0.1:
    print("  WARNING: Too many RESTAURANT categories!")
else:
    print("  OK: Low RESTAURANT usage in laptop")

print("\n" + "="*80)
print("TIER 1 IMPROVEMENTS COMPLETE")
print("="*80)


Prediction Verification

Restaurant test samples:

Sample 1:
  ID: rest26_asqp_test_1
  Text: My fiance had stewed chicken and I had the curried chicken- both were excellent
  Quadruplets: 1
  [1] A=curried chicken, C=FOOD#QUALITY, O=excellent, VA=7.83#7.83

Sample 2:
  ID: rest26_asqp_test_2
  Text: But the service , which is a huge reason to spend your hard-earned $ at a restaurant , was absolutely horrendous
  Quadruplets: 1
  [1] A=service, C=SERVICE#GENERAL, O=absolutely horrendous, VA=2.50#7.50

Sample 3:
  ID: rest26_asqp_test_3
  Text: Service was very nice but slow , especially since there were only 3 tables going the whole time we were there
  Quadruplets: 3
  [1] A=service, C=SERVICE#GENERAL, O=slow, VA=3.50#6.50
  [2] A=service, C=SERVICE#GENERAL, O=very nice, VA=6.50#6.50
  [3] A=service, C=SERVICE#GENERAL, O=slow, VA=3.50#6.50

--------------------------------------------------------------------------------
Laptop test samples:

Sample 1:
  ID: lap26_asqp_test_1
  Text: 